In [ ]:
# !pip install oracledb

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 10.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 17.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [oracledb]1/2 [oracledb]


In [10]:
import oracledb

# 일반적인 로컬 오라클 XE 환경 기준 설정 (본인 환경에 맞게 localhost 부분을 수정하세요)
dsn = "oracle-db:1521/XEPDB1"

try:
    conn = oracledb.connect(user="pknu", password="1234", dsn=dsn)
    cursor = conn.cursor()
    print("Oracle DB 연결 성공!")
except Exception as e:
    print(f"연결 실패: {e}")

Oracle DB 연결 성공!


In [11]:
# 기존 테이블 깔끔하게 날리기
try:
    cursor.execute("DROP TABLE employees")
    conn.commit()
    print("테이블 초기화 완료!")
except Exception as e:
    print(e)

테이블 초기화 완료!


In [12]:
def insert_employee(name, department):
    """새로운 직원 추가"""
    query = "INSERT INTO employees (name, department) VALUES (:1, :2)"
    cursor.execute(query, [name, department])
    conn.commit()

def select_employees():
    """전체 직원 조회"""
    query = "SELECT id, name, department FROM employees ORDER BY id"
    cursor.execute(query)
    return cursor.fetchall()

def update_employee(emp_id, department):
    """직원의 부서 정보 수정"""
    query = "UPDATE employees SET department = :1 WHERE id = :2"
    cursor.execute(query, [department, emp_id])
    conn.commit()

def delete_employee(emp_id):
    """직원 삭제"""
    query = "DELETE FROM employees WHERE id = :1"
    cursor.execute(query, [emp_id])
    conn.commit()

print("CRUD 실습 함수 정의 완료!")

CRUD 실습 함수 정의 완료!


In [14]:

cursor.execute("""
BEGIN
    EXECUTE IMMEDIATE 'CREATE TABLE employees (
        id NUMBER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
        name VARCHAR2(100),
        department VARCHAR2(50)
    )';
EXCEPTION
    WHEN OTHERS THEN
        IF SQLCODE != -955 THEN
            RAISE;
        END IF;
END;""")
conn.commit()

In [15]:
# 1. 데이터 삽입 테스트
insert_employee("Alice", "HR")
insert_employee("Bob", "IT")
employees = select_employees()
print("After insert:", employees)

# 삽입된 데이터가 있다면 실습 진행
if employees:
    # 현재 테이블에 존재하는 첫 번째 데이터의 실제 id와 두 번째 데이터의 실제 id를 가져옵니다.
    first_id = employees[0][0]
    
    # 2. 데이터 수정 테스트 (첫 번째 직원의 부서를 Finance로 변경)
    update_employee(first_id, "Finance")
    print("After update:", select_employees())

    # 3. 데이터 삭제 테스트 (방금 수정한 첫 번째 직원 삭제)
    delete_employee(first_id)
    print("After delete:", select_employees())

# 실습 종료 후 연결 닫기
cursor.close()
conn.close()
print("DB 연결 해제 완료.")

After insert: [(1, 'Alice', 'HR'), (2, 'Bob', 'IT')]
After update: [(1, 'Alice', 'Finance'), (2, 'Bob', 'IT')]
After delete: [(2, 'Bob', 'IT')]
DB 연결 해제 완료.
